In [1]:
import jams
import numpy as np
from pathlib import Path
from common import OUTPUT_DIM_NOTES
#pip install pyguitarpro
import guitarpro
from joblib import Parallel, delayed
import math
# Define the directory containing JAMS files. The corresponding input audio files have identical names but with .wav extension. These names are deduced
# from the jams files automatically.
# jams_dirs=['/home/gerald/guitarset/annotation/rock']
# Configuration

limit_gb = 3200
limit_bytes = limit_gb * (1024**3)
jams_dirs='/data/SynthTab/stratocaster_clean_bridge_jams'

#output_data_dir='/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices'
output_data_dir='/data/data_slices'
chunk_size=1 

def convert_period_s_to_frames(start_s,end_s,sample_rate):
    start_frame=int(math.floor(start_s*sample_rate))
    end_frame=int(math.ceil(end_s*sample_rate))
    return start_frame,end_frame

def load_jams_file(file_path):
    with open(file_path, 'r') as f:
        jams_data = jams.load(f,validate=False)
    return jams_data
# intervals += (60 / tempo) * num_ticks / guitarpro.Duration.quarterTime
#Extract from synthtab jams file the note_tab annotations
def extract_guitar_annotations(jams_data,duration_bin_s=1000):
    #Get the tempi from the jams file
    tempos=[]
    for annotation in jams_data.annotations:
        if annotation.namespace=='tempo':
            for obs in annotation.data:
                tempos.append((obs.time,obs.value)) #(time,bpm)

    # Sort the tempos by ticks
    tempos=sorted(tempos,key=lambda x:x[0])
    
    # Get the annotations. Since their times are in ticks, we need to convert them to seconds using the tempo map
    annotations=[]
    for annotation in jams_data.annotations:
        if annotation.namespace=='note_tab':
            base_tuning=annotation.sandbox["open_tuning"]
            data=annotation.data
            for note in data:
               
                
                #Convert from ticks to seconds
                ticks_per_beat=guitarpro.Duration.quarterTime
                note_begin_s=0
                note_end_s=0
                last_tick_jump=0
                # The mapping from ticks to seconds is a piecewise constant function on the tick axis, with discontinuities at the tempo change ticks.
                # integrate over tempos for the start time

                for tempo in tempos:
                    current_tempo=tempo[1]
                    seconds_per_tick=60/(current_tempo*ticks_per_beat)
                    if tempo[0]<note.time:
                        delta_ticks=tempo[0]-last_tick_jump
                        note_begin_s+=seconds_per_tick*delta_ticks
                        last_tick_jump=tempo[0]
                    
                delta_ticks=note.time-last_tick_jump
                note_begin_s+=seconds_per_tick*delta_ticks
                # integrate over tempos for the end time
                last_tick_jump=0
                noteend_ticks=note.time+note.duration
                for tempo in tempos:
                    current_tempo=tempo[1]
                    seconds_per_tick=60/(current_tempo*ticks_per_beat)
                    if tempo[0]<noteend_ticks:
                        delta_ticks=tempo[0]-last_tick_jump
                        note_end_s+=seconds_per_tick*delta_ticks
                        last_tick_jump=tempo[0]
                    
                delta_ticks=noteend_ticks-last_tick_jump
                note_end_s+=seconds_per_tick*delta_ticks
                        
                
                
                noteduration=note_end_s-note_begin_s

                annotations.append({
                    'time':note_begin_s,
                    'duration':noteduration,
                    'midi_note':note.value['fret']+base_tuning,
                    'confidence':note.confidence
                })
    return annotations
    # max_time=max(note['time']+note['duration'] for note in annotations)

    # num_bins=int(math.ceil(max_time/duration_bin_s))
    # binned_annotations=[[] for _ in range(num_bins)]
    # for note in annotations:
    #     bin_idx=int(note['time']//duration_bin_s)
    #     binned_annotations[bin_idx].append(note)
    # return binned_annotations,num_bins,max_time

def create_pianoroll(annotations,segment_start_s=None,segment_end_s=None, sr=48000, hop_length=256,  max_note=OUTPUT_DIM_NOTES,onsets_2d=False):
    max_time=max(note['time']+note['duration'] for note in annotations)
    frame_time=1/sr
    if segment_start_s is not None and segment_end_s is not None:
        
        segment_start_frame,segment_end_frame=convert_period_s_to_frames(segment_start_s,segment_end_s,sr)
        num_frames=segment_end_frame-segment_start_frame
        num_notes=max_note
        piano_roll=np.zeros((num_frames,num_notes))

        def process_note(note):
            #Check if note is within segment
            note_end_time=note['time']+note['duration']
            if note_end_time<segment_start_s or note['time']>segment_end_s:
                return
            note_start_time=max(note['time'],segment_start_s)
            note_end_time=min(note_end_time,segment_end_s)
            start_frame,end_frame=convert_period_s_to_frames(note_start_time,note_end_time,sr)
            start_frame=start_frame-segment_start_frame
            end_frame=min(end_frame-segment_start_frame,num_frames)
            note_idx=int(note['midi_note'])
            if note_idx>=0 and note_idx<num_notes:
                piano_roll[start_frame:end_frame,note_idx]=1
        Parallel(n_jobs=1,backend='threading')(delayed(process_note)(note) for note in annotations)


    else:
        num_frames=int(math.ceil(max_time *sr))
        num_notes=max_note
        piano_roll=np.zeros((num_frames,num_notes))
        def process_note(note):
            start_frame,end_frame=convert_period_s_to_frames(note['time'],note['time']+note['duration'],sr)
            note_idx=int(note['midi_note'])
            if note_idx>=0 and note_idx<num_notes:
                piano_roll[start_frame:end_frame,note_idx]=1

        Parallel(n_jobs=1,backend='threading')(delayed(process_note)(note) for note in annotations)           

                
    piano_roll=np.swapaxes(piano_roll,0,1)
    onsets=np.zeros_like(piano_roll)
    onsets[:,1:]=np.diff(piano_roll,axis=1)
    onsets=np.max(onsets,axis=0)
    onsets[np.where(onsets<0)[0]]=1
    if onsets_2d:
        onsets=np.tile(onsets,(OUTPUT_DIM_NOTES,1))
        
    for t in range(piano_roll.shape[1]):
        notes=piano_roll[range(0,OUTPUT_DIM_NOTES),t].max()
        if notes==0:
            piano_roll[OUTPUT_DIM_NOTES-1,t]=1

    return piano_roll,onsets

def load_jams_annotations(file_path, sr=48000, hop_length=256, max_note=OUTPUT_DIM_NOTES,onsets_2d=False):
    jams_data=load_jams_file(file_path)
    return extract_guitar_annotations(jams_data)


I0000 00:00:1770031609.743535   65140 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1770031609.767271   65140 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1770031610.204556   65140 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
from fretboard import FretBoard
from common import frame_size,reshape_to_nn_input,reshape_to_nn_output,save_data_slices,plot_heatmap
from IPython.display import display
import os
from os import path
from scipy import io
import soundfile as sf
import matplotlib.pyplot as plt
import glob 
import tensorflow as tf
import time
def _bytes_feature(value: bytes) -> tf.train.Feature:
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def serialize_example(input: np.ndarray, output: np.ndarray) -> bytes:
    feature = {
        "input":  _bytes_feature(input.tobytes()),
        "output": _bytes_feature(output.tobytes()),
    }
    example = tf.train.Example(features=tf.train.Features(feature=feature))
    return example.SerializeToString()

def write_tfrecord(path, input, output):
    assert input.shape[0] == output.shape[0]
    with tf.io.TFRecordWriter(path) as w:
        for i in range(input.shape[0]):
            ex = serialize_example(input[i], output[i])
            w.write(ex)
# Files where converted with for file in *.wav; do sox -SG "$file" -r 48000 -e float -b 32 48k/"$file"; done
def process_annotation_segment(annotations,audio,dirname,sample_rate,segment_start_s=None,segment_end_s=None):
        start_timing_segment=time.time()
        # Create output directories
        if segment_start_s is not None and segment_end_s is not None:
            segment_label=f"{int(segment_start_s)}s_to_{int(segment_end_s)}s"
            outdir=os.path.join(dirname,segment_label,'output')
            indir=os.path.join(dirname,segment_label,'input')
        else:
            outdir=os.path.join(dirname,'output')
            indir=os.path.join(dirname,'input')
        # return if both dirs exist
        if os.path.exists(outdir) and os.path.exists(indir):
            print(f"Data slices already exist for segment {segment_start_s} to {segment_end_s}, skipping")
            return
        # Compute the piano roll and onsets
        start_time=time.time()
        piano_roll,onsets=create_pianoroll(annotations,segment_start_s=segment_start_s,segment_end_s=segment_end_s,sr=sample_rate,hop_length=frame_size,max_note=OUTPUT_DIM_NOTES,onsets_2d=False)
        stop_time=time.time()
        print(f"Piano roll creation time: {stop_time-start_time:.2f} seconds")
        # We want to cut out sections around the onsets
        # Compute the positions to preserve
        start_time=time.time()
        # onset_positions=np.where(onsets>0)[0]
        # for onset in onset_positions:
        #     onsets[(onset-4*frame_size):onset+4*frame_size]=1

        # preserve_indices=np.where(onsets==0)[0]


        
        # Match piano roll length to audio length
        lenpianorool=piano_roll.shape[1]
        desired_length=len(audio)
        if lenpianorool<desired_length:
            # Pad with zeros
            padding = desired_length - lenpianorool
            piano_roll = np.pad(piano_roll, ((0, 0), (0, padding)), mode='constant')
        elif lenpianorool>desired_length:
            # Truncate
            piano_roll = piano_roll[:, :desired_length]
        end_time=time.time()
        print(f"Piano roll length adjustment time: {end_time-start_time:.2f} seconds")


        start_time=time.time()

        #Reshape audio to 2D (channels, frames)
        audio=np.expand_dims(audio,axis=0)
        print("Sizes after reshaping:"+str(audio.shape)+", "+str(piano_roll.shape))
        assert audio.shape[1]==piano_roll.shape[1],"Mismatched shapes"
        end_time=time.time()
        print(f"Onset pruning time: {end_time-start_time:.2f} seconds")
        # # Visualize the pruned data
        # display(plot_heatmap(piano_roll))
        # display(plot_heatmap(filterbank_out))

        # Reshape to NN input/output format
        start_time=time.time()
        input_slices=reshape_to_nn_input(audio)
        audio=[]
        output_slices=reshape_to_nn_output(piano_roll)
        piano_roll=[]
        end_time=time.time()
        print(f"Reshaping to NN input/output time: {end_time-start_time:.2f} seconds")

        
        
        Path(outdir).mkdir(parents=True,exist_ok=True)
        Path(indir).mkdir(parents=True,exist_ok=True)
        start_time=time.time()

        silent_range_step=frame_size*4
        ranges_to_preserve=np.zeros(output_slices.shape[0],dtype=bool)
        for i in range(0,output_slices.shape[0],silent_range_step):
            end=min(i+silent_range_step,output_slices.shape[0])
            if np.sum(output_slices[i:end,(OUTPUT_DIM_NOTES-1)])==0:
                ranges_to_preserve[i:end]=True
                # print(f"Marking  range for preservation: {i} to {i+silent_range_step}")

        percentage_preserved=100.0*np.sum(ranges_to_preserve)/ranges_to_preserve.shape[0]
        print(f"Preserving {percentage_preserved:.2f}% of data slices after removing silent ranges")
        end_time=time.time()
        print(f"Silent range removal time: {end_time-start_time:.2f} seconds")

        input_slices=(input_slices[ranges_to_preserve]).astype(np.float32)
        output_slices=(output_slices[ranges_to_preserve]).astype(np.int8)

        #create tensorflow dataset
        print("Creating tensorflow dataset")
        start_time=time.time()
        write_tfrecord(os.path.join(indir,'data.tfrecord'),input_slices,output_slices)
        end_time=time.time()
        print(f"TFRecord writing time: {end_time-start_time:.2f} seconds")
        end_timing_segment=time.time()
        print(f"Total time for segment processing: {end_timing_segment-start_timing_segment:.2f} seconds")
        
        # dataset=tf.data.Dataset.from_tensor_slices((input_slices,output_slices))
        # # save the data slices
        # tf.data.Dataset.save(dataset,indir)
        # print("Saving input data to ",indir)
        
        # save_data_slices(indir,input_slices,chunk_size)
        
        # print("Saving output data to ",outdir)
        
        # save_data_slices(outdir,output_slices,chunk_size)

def prepare_audio_midi_data(jams_file):

    # Compute the piano roll and onsets
    print(f"Processing {jams_file}")

    annotations=load_jams_annotations(jams_file)
    #Output directory
    # basename=os.path.basename(jams_file).replace('.jams','')
    jamsdirname=os.path.dirname(jams_file)
    #get the parent directory
    jamsdirname=os.path.relpath(jamsdirname,start=jams_dirs)
    dirname=os.path.join(output_data_dir,jamsdirname)
    
    #Input audio
    audio_file=jams_file.replace('.jams','_48k.flac')#path.join(input_audio,basename+'_mix.wav')
    print("Loading audio file ",audio_file)
    if os.path.exists(audio_file)==False:
        print("Audio file does not exist, skipping ",audio_file)
        return
    audio,sample_rate=sf.read(audio_file)

    # Normalize the audio
    maxaudio=np.max(np.abs(audio))
    audio=audio/maxaudio
    # plt.plot(audio)
    # plt.show()

    # Check the sample rate
    if sample_rate!=48000:
        print("Unexpected sample rate ",sample_rate)
        return
    audiolen_s=len(audio)/sample_rate
    print(f"Audio length: {audiolen_s:.2f} seconds")
    segment_duration_s=2*60
    # if audiolen_s>segment_duration_s:
    #     print("Processing in smaller segments")
    #     num_segments=int(math.ceil(audiolen_s/segment_duration_s))
    #     print(f"Number of segments: {num_segments}")
    #     for seg_idx in range(num_segments):
    #         segment_start_s=seg_idx*segment_duration_s
    #         segment_end_s=min((seg_idx+1)*segment_duration_s,audiolen_s)
    #         print(f"Processing segment {seg_idx+1}/{num_segments}: {segment_start_s:.2f}s to {segment_end_s:.2f}s")
    #         # Extract the audio segment
    #         start_sample,end_sample=convert_period_s_to_frames(segment_start_s,segment_end_s,sample_rate)
    #         end_sample=min(end_sample,len(audio))
    #         audio_segment=audio[start_sample:end_sample]
    #         process_annotation_segment(annotations,audio_segment,dirname,sample_rate,segment_start_s=segment_start_s,segment_end_s=segment_end_s)
    # else:
    process_annotation_segment(annotations,audio,dirname,sample_rate)
import subprocess
import sys

def get_size_with_du(path):
    try:
        # -s: summary (total for the path)
        # -b: output in bytes
        result = subprocess.check_output(['du', '-sb', path], stderr=subprocess.DEVNULL)
        # result is like b'536870912000\t/path/to/dir\n'
        return int(result.split()[0])
    except Exception as e:
        print(f"Error checking directory size: {e}")
        return 0

from joblib import Parallel, delayed

# Flatten files as shown above
all_jams_files = sorted(glob.glob(os.path.join(jams_dirs,'**', '*.jams'),recursive=True))# [f for d in jams_dirs for f in sorted(glob.glob(os.path.join(d,'**', '*.jams'),recursive=True))]

#Parallel(n_jobs=5)(delayed(prepare_audio_midi_data)(f) for f in all_jams_files) 



                
 
for jams_file in all_jams_files:
    prepare_audio_midi_data(jams_file)
    current_size = get_size_with_du(output_data_dir)
    print(f"Current dataset size: {current_size / (1024**3):.2f} GB")
    if current_size >= limit_bytes:
        print(f"Reached size limit of {limit_gb} GB, stopping data preparation.")
        break

    
            

    


Processing /data/SynthTab/stratocaster_clean_bridge_jams/(HED)P - E -  - I Got You - gp4__3 - Electric Jazz Guitar__midi/stratocaster_clean_bridge_nonoise_stere.jams
Loading audio file  /data/SynthTab/stratocaster_clean_bridge_jams/(HED)P - E -  - I Got You - gp4__3 - Electric Jazz Guitar__midi/stratocaster_clean_bridge_nonoise_stere_48k.flac
Audio length: 162.00 seconds
Piano roll creation time: 33.86 seconds
Piano roll length adjustment time: 0.53 seconds
Sizes after reshaping:(1, 7776000), (129, 7776000)
Onset pruning time: 0.00 seconds
reshape data by a factor of 256...
(1, 7776000)
Reshaped the output data to  
(30375, 1, 256)
reshape data by a factor of 256...
(129, 7776000)
Reshaped the output data to  
(30375, 129)
Reshaping to NN input/output time: 0.50 seconds
Preserving 2.24% of data slices after removing silent ranges
Silent range removal time: 0.00 seconds
Creating tensorflow dataset
TFRecord writing time: 0.00 seconds
Total time for segment processing: 35.25 seconds
Curre

KeyboardInterrupt: 

In [ ]:
import numpy as np
test=np.zeros((3,16))
# fill test with values 0 to 47
for i in range(3):
    for j in range(16):
        test[i,j]=i*16+j
print(test)
preserve_indices=[0,2,4,6,8,10,12,14]
test=np.take(test,preserve_indices,axis=1,mode='clip')


print(test)

test2=test.reshape((3,4,2))
print(test2)

print(test2.shape)
print(np.max(test2,axis=2))

[[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]
 [16. 17. 18. 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31.]
 [32. 33. 34. 35. 36. 37. 38. 39. 40. 41. 42. 43. 44. 45. 46. 47.]]
[[ 0.  2.  4.  6.  8. 10. 12. 14.]
 [16. 18. 20. 22. 24. 26. 28. 30.]
 [32. 34. 36. 38. 40. 42. 44. 46.]]
[[[ 0.  2.]
  [ 4.  6.]
  [ 8. 10.]
  [12. 14.]]

 [[16. 18.]
  [20. 22.]
  [24. 26.]
  [28. 30.]]

 [[32. 34.]
  [36. 38.]
  [40. 42.]
  [44. 46.]]]
(3, 4, 2)
[[ 2.  6. 10. 14.]
 [18. 22. 26. 30.]
 [34. 38. 42. 46.]]
